In [0]:
%sql

create schema if not exists arshprojects.silver;

In [0]:
bronze_table = 'arshprojects.bronze.patient_raw'
silver_table = 'arshprojects.silver.dim_patient'
checkpoint_path = "abfss://data@arshprojectsstorage.dfs.core.windows.net/silver/dim_patient/checkpoint/"

In [0]:
from pyspark.sql.functions import sha2, col, current_timestamp, monotonically_increasing_id, concat_ws

In [0]:
%sql

select * from arshprojects.bronze.patient_raw;

In [0]:
df_patient_bronze = (
    spark.readStream.table(bronze_table)
)

df_patient_clean = (
    df_patient_bronze
        .dropDuplicates(["patient_id"])
        .withColumn("patient_first_last_name_masked", sha2(concat_ws('|',col("first_name"),col("last_name")), 256))
        .withColumn("load_timestamp", current_timestamp())
)



In [0]:
from delta.tables import DeltaTable

def merge_dim_patient(batch_df, batch_id):
    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        return

    # Load Delta table by name and upsert
    dim_patient = DeltaTable.forName(spark, silver_table)

    (dim_patient.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.patient_id = s.patient_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
(
    df_patient_clean.writeStream
        .foreachBatch(merge_dim_patient)
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)


In [0]:
%sql

select * from arshprojects.silver.dim_patient;